[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/S12_03_nlp_clasico.ipynb)

# S12 · 03 · NLP clásico: sentimiento y temas en las reseñas de TurisData

**Tiempo estimado:** 1 h 10 min · **Sprint 12** (UD 4.8) · Cliente: **TurisData Canarias**

**Lo que vas a aprender**
1. **Limpiar y tokenizar** texto en español.
2. Convertir texto en números: **bolsa de palabras** y **TF-IDF**.
3. Clasificar el **sentimiento** con regresión logística y Naive Bayes, y leer qué palabras pesan.
4. Detectar **temas** con palabras clave y descubrir expresiones frecuentes con **n-gramas**.
5. Reconocer los **límites** de estos métodos (negaciones, ironía).

**Contexto.** TurisData recibe 1.500 reseñas. La dirección no puede leerlas todas: quiere saber
*qué opina la gente* (sentimiento) y *de qué habla* (temas).

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import re, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score

warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (8, 3.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

resenas = pd.read_csv(dato("resenas.csv"))
print(resenas.shape)
resenas.head()

## 1. Conocer los datos
Cada reseña tiene texto, nota (1-5), sentimiento (positivo / neutro / negativo) y temas mencionados.
**Aviso:** son reseñas *sintéticas* (inventadas para el curso): más repetitivas y fáciles que las reales.

In [ ]:
print(resenas["sentimiento"].value_counts(normalize=True).round(3).to_string())
resenas["longitud"] = resenas["texto"].str.split().str.len()
print("Palabras por reseña (media):", round(resenas["longitud"].mean(), 1))
for t in resenas.sample(3, random_state=SEMILLA)["texto"]:
    print("-", t)

## 2. Limpieza y tokenización
**Idea.** El ordenador ve "Bueno", "bueno!" y "BUENO" como palabras distintas. La limpieza las unifica:
minúsculas, quitar signos y separar en **tokens** (palabras). Luego se quitan las **palabras vacías**
(*stopwords*: "de", "el", "la"...), que casi no aportan. Ojo: **no quitamos "no"**, porque cambia el sentido.

**Ejercicio 1.** Completa `limpiar(texto)`: pasa a minúsculas, cambia todo lo que no sean letras (incluidas tildes y ñ) o espacios por un espacio,
y colapsa espacios repetidos. Debe cumplir `limpiar("¡Qué BUENO!...  Sí") == "qué bueno sí"`.

In [ ]:
def limpiar(texto):
    # TODO: texto.lower(), re.sub(r"[^a-záéíóúüñ\s]", " ", ...) y luego " ".join(texto.split())
    ...

assert limpiar("¡Qué BUENO!...  Sí") == "qué bueno sí", "Revisa limpiar: minúsculas, signos fuera y espacios únicos."
assert limpiar("Precio 3€, ok") == "precio ok", "Los números y símbolos también se eliminan."

In [ ]:
PALABRAS_VACIAS = {"a", "al", "con", "de", "del", "el", "en", "es", "esta", "estaba", "era", "fue", "la", "las", "lo", "los",
                   "me", "mi", "muy", "nos", "para", "por", "que", "se", "su", "un", "una", "y", "o", "u", "e"}

def tokenizar(texto, quitar_vacias=True):
    tokens = limpiar(texto).split()
    return [t for t in tokens if not (quitar_vacias and t in PALABRAS_VACIAS)]

print(tokenizar("La limpieza de la habitación era impecable!"))
resenas["limpio"] = resenas["texto"].apply(limpiar)

**Ejercicio 2.** ¿Cuáles son las 5 palabras (tokens, ya sin palabras vacías) más frecuentes de todo el corpus?
Guarda en `top5` una lista de 5 tuplas `(palabra, veces)`. Pista: `collections.Counter` y `.most_common(5)`.

In [ ]:
from collections import Counter
# TODO: Counter sobre todos los tokens de resenas["texto"]
...
print(top5)
assert len(top5) == 5 and all(len(x) == 2 for x in top5), "top5 debe ser una lista de 5 pares (palabra, veces)."
assert top5[0][1] >= top5[4][1], "Ordénalas de mayor a menor frecuencia."
assert all(p not in PALABRAS_VACIAS for p, _ in top5), "Quita las palabras vacías con tokenizar()."

## 3. De texto a números: bolsa de palabras y TF-IDF
**Bolsa de palabras (BoW):** una columna por palabra del vocabulario; cada reseña cuenta cuántas veces aparece.
Se pierde el orden, pero es muy útil.

**TF-IDF:** igual, pero las palabras que aparecen en *todas* las reseñas pesan menos y las **distintivas** pesan más
(*term frequency × inverse document frequency*).

In [ ]:
ejemplo = ["el desayuno era escaso", "el desayuno merece la pena", "la habitación estaba sucia"]
bow = CountVectorizer()
M = bow.fit_transform(ejemplo)
print(pd.DataFrame(M.toarray(), columns=bow.get_feature_names_out(), index=["r1", "r2", "r3"]))
tf = TfidfVectorizer()
M2 = tf.fit_transform(ejemplo)
print(pd.DataFrame(M2.toarray().round(2), columns=tf.get_feature_names_out(), index=["r1", "r2", "r3"]))

Observa que "desayuno" (aparece en dos de tres) pesa menos que "sucia" (solo en una) en TF-IDF.

## 4. Clasificar el sentimiento
Partimos en entrenamiento y prueba (estratificado para mantener las proporciones) y **ajustamos el vectorizador solo con el entrenamiento**
(misma idea que el escalado sin fuga). Usamos `Pipeline` para no equivocarnos.

In [ ]:
from sklearn.pipeline import make_pipeline

X_ent, X_test, y_ent, y_test = train_test_split(resenas["texto"], resenas["sentimiento"], test_size=0.25,
                                                stratify=resenas["sentimiento"], random_state=SEMILLA)
print(len(X_ent), "para entrenar,", len(X_test), "para probar")

**Ejercicio 3.** Crea y entrena dos modelos y guarda su exactitud en prueba:
- `modelo_lr`: `TfidfVectorizer(preprocessor=limpiar)` + `LogisticRegression(max_iter=1000)` → `acc_lr`
- `modelo_nb`: `CountVectorizer(preprocessor=limpiar)` + `MultinomialNB()` → `acc_nb`

Usa `make_pipeline(vectorizador, clasificador)` y `.fit(X_ent, y_ent)`.

In [ ]:
# TODO: make_pipeline(TfidfVectorizer(preprocessor=limpiar), LogisticRegression(max_iter=1000)) y análogo para NB
...
print(f"Regresión logística + TF-IDF : {acc_lr:.3f}")
print(f"Naive Bayes + bolsa de palabras: {acc_nb:.3f}")
assert acc_lr > 0.7 and acc_nb > 0.7, "Las dos exactitudes deberían superar 0.7. Revisa el pipeline y que uses X_ent/y_ent para entrenar."
baseline = y_test.value_counts(normalize=True).max()
print(f"(Un clasificador que siempre dice la clase mayoritaria acertaría {baseline:.3f})")

In [ ]:
print(classification_report(y_test, modelo_lr.predict(X_test)))
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, modelo_lr.predict(X_test), ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión (regresión logística)"); ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.grid(False)
plt.show()

### ¿Qué palabras pesan?
La gran ventaja de los modelos clásicos es que se pueden **explicar**: cada palabra tiene un peso hacia cada clase.

In [ ]:
vec, clf = modelo_lr.named_steps["tfidfvectorizer"], modelo_lr.named_steps["logisticregression"]
vocab = np.array(vec.get_feature_names_out())
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, clase, color in zip(ejes, ["positivo", "negativo"], ["#3b8f5a", "#c0463a"]):
    i = list(clf.classes_).index(clase)
    top = np.argsort(clf.coef_[i])[-8:]
    ax.barh(vocab[top], clf.coef_[i][top], color=color)
    ax.set_title(f"Palabras que más empujan a «{clase}»"); ax.set_xlabel("Peso en el modelo")
plt.tight_layout(); plt.show()

## 5. Los límites: negaciones
Una bolsa de palabras no entiende el orden. Probemos frases nuevas que el modelo no ha visto.

In [ ]:
pruebas = ["La limpieza es impecable", "La limpieza no es impecable", "El personal fue borde", "El personal no fue borde",
           "Vistas increíbles pero un ruido constante"]
for f, p in zip(pruebas, modelo_lr.predict(pruebas)):
    print(f"{p:9s} ← {f}")

**Ejercicio 4.** Observa los resultados y escribe en `falla_negacion` (True/False) si el modelo trata igual las frases
"con negación" que las "sin negación" (es decir, ¿se equivoca con "no es impecable"?). Después, en `bigrama_util`, escribe
(texto) por qué usar **bigramas** (`ngram_range=(1, 2)`) podría ayudar en una frase.
Pista: los bigramas conservan pares como "no es".

In [ ]:
# TODO: compara predicción de la 1.ª y 2.ª frase; explica que un bigrama como «no fue» mantiene la negación junto a la palabra
...
assert isinstance(falla_negacion, (bool, np.bool_)), "falla_negacion debe ser True o False."
assert len(bigrama_util) > 30, "Escribe una explicación de al menos una frase en bigrama_util."

## 6. Temas por palabras clave
**Idea.** Sin entrenar nada: definimos, para cada tema, una lista de palabras que lo delatan. Si la reseña contiene alguna, se asigna el tema.
Es una técnica **transparente y barata**; su límite es que hay que mantener la lista.
Comparamos con la columna `temas` (etiquetas del corpus) usando la puntuación F1 de cada tema.

In [ ]:
TEMAS = {
    "limpieza":  ["limpi", "sucia", "sucio", "pelos", "sábanas", "toallas"],
    "ruido":     ["ruid", "silencio", "música", "dormir", "oye", "tranquilo"],
    "desayuno":  ["desayuno", "café", "buffet", "colas"],
    "personal":  ["personal", "recepción", "recepcionista", "trato", "borde", "atendía", "ayudó", "tardaron"],
    "ubicacion": ["ubicación", "situado", "lejos", "centro", "playa", "vistas", "coche", "zona"],
    "precio":    ["precio", "caro", "cuesta", "extras", "cobran", "compensa"],
}

def detectar_temas(texto, diccionario=TEMAS):
    t = limpiar(texto)
    return sorted(tema for tema, claves in diccionario.items() if any(c in t for c in claves))

resenas["temas_detectados"] = resenas["texto"].apply(detectar_temas)
resenas["temas_reales"] = resenas["temas"].fillna("").apply(lambda s: [x for x in s.split(",") if x])

def f1_tema(tema):
    real = resenas["temas_reales"].apply(lambda l: tema in l)
    pred = resenas["temas_detectados"].apply(lambda l: tema in l)
    return f1_score(real, pred)

f1_por_tema = pd.Series({t: f1_tema(t) for t in TEMAS}).sort_values()
print(f1_por_tema.round(2).to_string())

**Ejercicio 5.** ¿Cuál es el tema con peor F1 (`tema_peor`, texto) y cuántas reseñas del corpus mencionan **realmente** el tema `precio`
(`n_precio`, entero, usando `temas_reales`)?

Después mira los casos donde falla ese tema y decide qué cambio harías en el diccionario (¿quitar una clave demasiado amplia? ¿añadir otra?).

In [ ]:
# TODO: idxmin de f1_por_tema; contar cuántas listas de temas_reales contienen "precio"
...
print("Tema peor:", tema_peor, "| n precio:", n_precio)
assert tema_peor in TEMAS, "tema_peor debe ser una de las claves de TEMAS."
assert 300 < n_precio < 700, "n_precio: cuenta las reseñas cuya lista temas_reales contiene «precio»."

In [ ]:
errores = resenas[resenas["temas_reales"].apply(lambda l: tema_peor in l) != resenas["temas_detectados"].apply(lambda l: tema_peor in l)]
print(f"{len(errores)} reseñas donde falla «{tema_peor}». Ejemplos:")
for _, fila in errores.head(5).iterrows():
    print("-", fila["texto"], "| real:", fila["temas_reales"], "| detectado:", fila["temas_detectados"])

## 7. n-gramas: expresiones que se repiten
Un **n-grama** es una secuencia de n palabras seguidas. Los bigramas y trigramas más frecuentes en las reseñas **negativas**
revelan quejas concretas que una sola palabra esconde ("no compensa el precio").

In [ ]:
# Palabras de "relleno" que abren muchas reseñas y no dicen nada de la estancia
RELLENO = PALABRAS_VACIAS | {"pasamos", "semana", "aquí", "escapada", "fin", "viajamos", "familia", "fuimos", "trabajo", "general", "todo"}

def sin_relleno(texto):
    return " ".join(t for t in limpiar(texto).split() if t not in RELLENO)

def top_ngramas(textos, n=(2, 3), k=8):
    cv = CountVectorizer(preprocessor=sin_relleno, ngram_range=n, min_df=5)
    M = cv.fit_transform(textos)
    frec = pd.Series(np.asarray(M.sum(axis=0)).ravel(), index=cv.get_feature_names_out())
    return frec.sort_values(ascending=False).head(k)

neg = resenas.loc[resenas["sentimiento"] == "negativo", "texto"]
print(top_ngramas(neg).to_string())

## 8. Mini-reto integrador: panel de quejas
Construye `quejas`, una serie con el **número de reseñas negativas por tema** (usa `temas_detectados` y solo reseñas con `sentimiento == "negativo"`),
ordenada de mayor a menor. Dibuja un gráfico de barras y guarda en `queja_principal` el tema más frecuente.

In [ ]:
# TODO: explode() sobre temas_detectados de las negativas, value_counts()
...
fig, ax = plt.subplots()
quejas.sort_values().plot.barh(ax=ax, color="#c0463a")
ax.set_title("Reseñas negativas por tema"); ax.set_xlabel("Número de reseñas")
plt.show()
assert quejas.is_monotonic_decreasing, "quejas debe estar ordenada de mayor a menor."
assert queja_principal in TEMAS, "queja_principal debe ser un tema del diccionario."
assert quejas.sum() >= 300, "Cuenta solo reseñas negativas, un tema por fila (usa explode)."

## Preguntas de reflexión
1. La exactitud es muy alta. ¿Por qué no debemos fiarnos tanto de ese número con reseñas **sintéticas** y repetitivas?
2. ¿Qué ventaja tiene un modelo que puedes explicar (pesos de palabras) frente a uno que no?
3. ¿Qué haría falta para manejar ironía ("¡Qué gran idea poner música a las 3 de la mañana!")?
4. Las reseñas pueden contener nombres o datos personales: ¿qué harías antes de guardarlas y analizarlas?

## Resumen: qué has aprendido
- Limpiar, tokenizar y quitar palabras vacías (sin quitar "no").
- Bolsa de palabras y TF-IDF convierten texto en números; se ajustan **solo con el entrenamiento**.
- Regresión logística y Naive Bayes clasifican bien y se pueden explicar; fallan con negaciones.
- Las palabras clave dan temas de forma transparente; los n-gramas muestran quejas concretas.